**Resumen de Hallazgos del EDA**

El análisis se realizó sobre un dataset de marketing bancario que contiene 41,188 registros y 21 variables. Los puntos clave identificados son:

- Integridad de los Datos: El dataset está bastante limpio. No se detectaron valores nulos en ninguna de las 21 columnas. Solo se identificaron 12 registros duplicados, lo cual es mínimo considerando el tamaño de la muestra.

- Perfil de los Clientes: La variable age (edad) es de tipo entero, y el nivel educativo (education) se perfila como un factor relevante para el análisis de suscripción.

- Variable Objetivo (Target): La columna y indica si el cliente se suscribió a un depósito a plazo. Se observó que el nivel educativo influye en la decisión, con visualizaciones que comparan las suscripciones ("yes" / "no") según el grado de instrucción.

- Distribución de Edad: Se utilizó un diagrama de caja (boxplot) para analizar cómo se distribuye la edad en relación con el éxito de la campaña (y). Esto ayuda a identificar si existe un rango de edad específico más propenso a aceptar el producto.

- Variables Económicas: El dataset incluye indicadores macroeconómicos importantes como euribor3m (tasa Euribor a 3 meses), cons.price.idx (índice de precios al consumidor) y nr.employed (número de empleados), que permiten analizar el contexto económico de las llamadas.

**1. Tratamiento de Nulos e Imputación**

Aunque vimos que no hay nulos "clásicos" (NaN), en este dataset los nulos suelen venir como la categoría "unknown".

Primero, debemos convertir esos "unknown" a valores que Python reconozca como nulos para poder imputarlos.

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder

# Cargamos y reemplazamos
df = pd.read_csv('../data/raw/03-bank_marketing.csv', sep=';')
df.replace('unknown', np.nan, inplace=True)

# Veamos cuántos nulos reales tenemos ahora
print(df.isnull().sum())

age                  0
job                330
marital             80
education         1731
default           8597
housing            990
loan               990
contact              0
month                0
day_of_week          0
duration             0
campaign             0
pdays                0
previous             0
poutcome             0
emp.var.rate         0
cons.price.idx       0
cons.conf.idx        0
euribor3m            0
nr.employed          0
y                    0
dtype: int64


**Clasificación y Estrategia**

MCAR (Completamente al azar): marital, housing, loan, education. Son pocos nulos, usaremos Moda.

MAR (Al azar): job. Suelen depender de la edad o el nivel socioeconómico. Usaremos KNNImputer.

MNAR (No al azar): default. Es probable que la gente con deudas no quiera decirlo.

In [2]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder

# --- PASO A: MCAR (Imputación por Moda) ---
# Añadimos 'education' a este grupo
columnas_moda = ['marital', 'housing', 'loan', 'education']
imputer_moda = SimpleImputer(strategy='most_frequent')
df[columnas_moda] = imputer_moda.fit_transform(df[columnas_moda])

# --- PASO B: PREPARACIÓN PARA MAR Y MNAR ---
# Ahora solo 'job' y 'default' necesitan codificación para imputación compleja
encoder = OrdinalEncoder()
columnas_complejas = ['job', 'default']
df[columnas_complejas] = encoder.fit_transform(df[columnas_complejas])

# --- PASO C: MAR (Falta al azar) ---
# Usamos KNNImputer SOLO para 'job'
knn_imputer = KNNImputer(n_neighbors=5)
df[['job']] = knn_imputer.fit_transform(df[['job']])

# --- PASO D: MNAR (Falta no aleatoria) ---
# Seguimos usando IterativeImputer para 'default'
it_imputer = IterativeImputer(max_iter=10, random_state=42)
df[['default']] = it_imputer.fit_transform(df[['default']])

# --- PASO E: LIMPIEZA FINAL ---
# Redondeamos 'job' y 'default' y los pasamos a enteros
df[columnas_complejas] = df[columnas_complejas].round().astype(int)

# Invertimos el encoding de las columnas complejas para recuperar los textos originales
# Esto es importante para que el Punto 5 (Encoding final) funcione correctamente
df[columnas_complejas] = encoder.inverse_transform(df[columnas_complejas])

# Verificación
print("Nulos por columna después del tratamiento:")
print(df.isnull().sum())

Nulos por columna después del tratamiento:
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64


**2. Detectar y tratar outliers (valores atípicos)**

In [3]:
# Seleccionamos solo las columnas numéricas
num_cols = df.select_dtypes(include=[np.number]).columns

# Creamos un reporte de outliers
outlier_report = []

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    
    n_outliers = df[(df[col] < lim_inf) | (df[col] > lim_sup)].shape[0]
    pct_outliers = (n_outliers / len(df)) * 100
    
    outlier_report.append({'Columna': col, 'Outliers': n_outliers, 'Porcentaje': f"{pct_outliers:.2f}%"})

# Mostramos el resultado como una tabla limpia
df_reporte = pd.DataFrame(outlier_report)
print(df_reporte)

          Columna  Outliers Porcentaje
0             age       469      1.14%
1        duration      2963      7.19%
2        campaign      2406      5.84%
3           pdays      1515      3.68%
4        previous      5625     13.66%
5    emp.var.rate         0      0.00%
6  cons.price.idx         0      0.00%
7   cons.conf.idx       447      1.09%
8       euribor3m         0      0.00%
9     nr.employed         0      0.00%


Análisis de los resultados (Diagnóstico)

- duration (7.19%) y campaign (5.84%): Son los "sospechosos habituales". Hay llamadas muy largas y clientes contactados demasiadas veces. Estos deforman la media y deben ser tratados.

- previous (13.66%) y pdays (3.68%): Aunque el reporte dice que son outliers, en este dataset estas columnas indican si hubo contacto previo. La mayoría de los valores son 0 o 999 (nunca contactado). No tratar como outliers, porque son valores con un significado específico en el negocio.

- age (1.14%) y cons.conf.idx (1.09%): Tienen muy pocos. Son valores reales (ancianos o momentos de crisis económica). Un tratamiento suave es suficiente.

- job, education, default: Salen en 0 o casi 0 porque las convertimos a números fijos (ordinales), así que no tienen sentido como outliers numéricos.

In [4]:
from scipy.stats import zscore
from scipy.stats.mstats import winsorize

# 1. TRATAMIENTO PARA 'duration' -> Winsorizing
# Para limitar el impacto de llamadas extremadamente largas (7.19% outliers) sin eliminar los registros.
df['duration'] = winsorize(df['duration'], limits=[0.0, 0.05])


# 2. TRATAMIENTO PARA 'campaign' -> IQR Capping
#Para establecer un tope lógico al número de contactos y evitar sesgos por clientes excesivamente contactados.
Q1 = df['campaign'].quantile(0.25)
Q3 = df['campaign'].quantile(0.75)
IQR = Q3 - Q1
limite_superior_cap = Q3 + 1.5 * IQR
# Aplicamos el "techo" (capping)
df.loc[df['campaign'] > limite_superior_cap, 'campaign'] = limite_superior_cap


# 3. TRATAMIENTO PARA 'age' -> Z-Score Adjustment
# Para manejar de forma precisa la distribución de edades extremas.
df['age_z'] = np.abs(zscore(df['age']))
limite_edad_aceptable = df[df['age_z'] <= 3]['age'].max()
# Ajustamos solo los casos extremos
df.loc[df['age_z'] > 3, 'age'] = limite_edad_aceptable
# Borramos la columna auxiliar
df.drop(columns=['age_z'], inplace=True)

print("Tratamiento estratégico de outliers finalizado.")

Tratamiento estratégico de outliers finalizado.


In [5]:
# --- VALIDACIÓN DE OUTLIERS ---
print(f"1. Filas totales (Shape): {df.shape[0]}")
print(f"2. Nulos totales: {df.isnull().sum().sum()}")
print("\n3. Verificación de 'Techos' (Máximos):")
print(df[['age', 'duration', 'campaign']].max())

1. Filas totales (Shape): 41188
2. Nulos totales: 0

3. Verificación de 'Techos' (Máximos):
age          71
duration    753
campaign      6
dtype: int64


**3. Eliminar duplicados y registros inconsistentes.**

In [6]:
# --- A. DUPLICADOS ---
# Primero contamos cuántos hay para el informe
cant_duplicados = df.duplicated().sum()
print(f"Se encontraron {cant_duplicados} registros duplicados.")

# Eliminamos los duplicados y mantenemos solo la primera aparición
df.drop_duplicates(inplace=True)

# --- B. INCONSISTENCIAS ---
# En este dataset, verificamos que no existan duraciones o edades negativas
df = df[df['age'] > 0]
df = df[df['duration'] >= 0]

# --- C. REORGANIZACIÓN ---
# Al borrar filas, los números de las filas (index) quedan con huecos. 
# Re-ordenamos para que vayan de 0 en adelante sin saltos.
df.reset_index(drop=True, inplace=True)

print(f"Limpieza completada. Filas actuales: {df.shape[0]}")

Se encontraron 18 registros duplicados.
Limpieza completada. Filas actuales: 41170


Se realizó una depuración del dataset eliminando 18 registros duplicados que presentaban información idéntica en todas sus variables, evitando así el sobreajuste del modelo. Asimismo, se validó la consistencia de las variables críticas (age, duration), asegurando que no existan valores negativos. Se mantuvo el valor '999' en la variable pdays tras confirmar que es un código de negocio y no una inconsistencia técnica.

**4. Corregir tipos de datos**

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41170 entries, 0 to 41169
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41170 non-null  int64  
 1   job             41170 non-null  str    
 2   marital         41170 non-null  str    
 3   education       41170 non-null  str    
 4   default         41170 non-null  str    
 5   housing         41170 non-null  str    
 6   loan            41170 non-null  str    
 7   contact         41170 non-null  str    
 8   month           41170 non-null  str    
 9   day_of_week     41170 non-null  str    
 10  duration        41170 non-null  int64  
 11  campaign        41170 non-null  int64  
 12  pdays           41170 non-null  int64  
 13  previous        41170 non-null  int64  
 14  poutcome        41170 non-null  str    
 15  emp.var.rate    41170 non-null  float64
 16  cons.price.idx  41170 non-null  float64
 17  cons.conf.idx   41170 non-null  float64
 1

In [8]:
# 1. Convertir variables numéricas a Enteros (int)
# Esto quita los decimales ".0" de columnas que no los necesitan
cols_numericas = ['age', 'duration', 'campaign', 'pdays', 'previous']
df[cols_numericas] = df[cols_numericas].astype(int)

# 2. Convertir variables de texto a Tipo Categoría
# Seleccionamos todas las columnas que tienen palabras
cols_categoricas = ['job', 'marital', 'education', 'default', 'housing', 
                    'loan', 'contact', 'month', 'day_of_week', 'poutcome', 'y']

for col in cols_categoricas:
    df[col] = df[col].astype('category')

# 3. Verificación: Esto te mostrará la lista de columnas y su nuevo tipo
print(df.dtypes)

age                  int64
job               category
marital           category
education         category
default           category
housing           category
loan              category
contact           category
month             category
day_of_week       category
duration             int64
campaign             int64
pdays                int64
previous             int64
poutcome          category
emp.var.rate       float64
cons.price.idx     float64
cons.conf.idx      float64
euribor3m          float64
nr.employed        float64
y                 category
dtype: object


Se realizó una estandarización de los tipos de datos para garantizar la integridad técnica del dataset antes del modelado.

- Primero, se convirtieron las variables como age y duration al tipo integer (entero), eliminando los decimales generados durante la imputación de nulos.

- Segundo, se transformaron las variables cualitativas al tipo category. Esta acción optimiza el uso de memoria de la sesión y prepara las columnas para la correcta aplicación de algoritmos de codificación (Encoding) en la siguiente fase."

**5. Aplicar encoding a categoricas: OneHot, Ordinal, Target.**

In [9]:
# --- PUNTO 5: ENCODING INTEGRAL ---

# 1. Ordinal Encoding para 'education'
educacion_mapping = {
    'basic.4y': 1, 'basic.6y': 2, 'basic.9y': 3, 
    'high.school': 4, 'professional.course': 5, 'university.degree': 6,
    'unknown': 0
}
df['education'] = df['education'].astype(str).str.strip().map(educacion_mapping)

# 2. Label Encoding para la variable objetivo 'y' (1/0)
df['y'] = df['y'].astype(str).str.strip().map({'yes': 1, 'no': 0})

# 3. One-Hot Encoding para el resto de categorías nominales
# Estas no tienen un orden específico
cols_dummies = [
    'job', 'marital', 'default', 'housing', 'loan', 
    'contact', 'month', 'day_of_week', 'poutcome'
]

# Creamos las variables ficticias (0 y 1)
df = pd.get_dummies(df, columns=cols_dummies, drop_first=True)

# 4. Convertimos posibles remanentes booleanos a enteros
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

# Verificación de que todo es numérico
print("=== RESUMEN DE TRANSFORMACIÓN ===")
print(f"Nuevas dimensiones: {df.shape}")
print(df.dtypes.value_counts())

=== RESUMEN DE TRANSFORMACIÓN ===
Nuevas dimensiones: (41170, 43)
int64      37
float64     6
Name: count, dtype: int64


In [10]:
df.head()

,age,education,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,56,1.0,261,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0
1,57,4.0,149,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0
2,37,4.0,226,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0
3,40,2.0,151,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0
4,56,4.0,307,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41170 entries, 0 to 41169
Data columns (total 43 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age                   41170 non-null  int64  
 1   education             41152 non-null  float64
 2   duration              41170 non-null  int64  
 3   campaign              41170 non-null  int64  
 4   pdays                 41170 non-null  int64  
 5   previous              41170 non-null  int64  
 6   emp.var.rate          41170 non-null  float64
 7   cons.price.idx        41170 non-null  float64
 8   cons.conf.idx         41170 non-null  float64
 9   euribor3m             41170 non-null  float64
 10  nr.employed           41170 non-null  float64
 11  y                     41170 non-null  int64  
 12  job_blue-collar       41170 non-null  int64  
 13  job_entrepreneur      41170 non-null  int64  
 14  job_housemaid         41170 non-null  int64  
 15  job_management        41170 no

In [12]:
reserva=df.copy()

In [13]:
reserva

,age,education,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,56,1.0,261,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0
1,57,4.0,149,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0
2,37,4.0,226,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0
3,40,2.0,151,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0
4,56,4.0,307,1,999,0,1.1,93.994,-36.4,4.857,...,1,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41165,71,5.0,334,1,999,0,-1.1,94.767,-50.8,1.028,...,0,1,0,0,0,0,0,0,1,0
41166,46,5.0,383,1,999,0,-1.1,94.767,-50.8,1.028,...,0,1,0,0,0,0,0,0,1,0
41167,56,6.0,189,2,999,0,-1.1,94.767,-50.8,1.028,...,0,1,0,0,0,0,0,0,1,0
41168,44,5.0,442,1,999,0,-1.1,94.767,-50.8,1.028,...,0,1,0,0,0,0,0,0,1,0


**6. Escalamiento de datos**

In [14]:
from sklearn.preprocessing import StandardScaler

# 1. Seleccionamos las columnas numéricas originales (no consideramos el target)
cols_a_escalar = [
    'age', 'duration', 'campaign', 'pdays', 'previous', 
    'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'
]

scaler = StandardScaler()

# 2. Aplicamos el escalamiento
df[cols_a_escalar] = scaler.fit_transform(df[cols_a_escalar])

# 3. Verificación final del dataset
print("=== PRE-PROCESAMIENTO FINALIZADO ===")
print(f"Dimensiones finales: {df.shape}")
print("\nEstadísticos de las columnas escaladas (Media debe ser cercana a 0):")
print(df[cols_a_escalar].describe().round(2).loc[['mean', 'std']])
# Guardar el dataset limpio para el modelo
# df.to_csv('bank_marketing_cleaned.csv', index=False)

=== PRE-PROCESAMIENTO FINALIZADO ===
Dimensiones finales: (41170, 43)

Estadísticos de las columnas escaladas (Media debe ser cercana a 0):
      age  duration  campaign  pdays  previous  emp.var.rate  cons.price.idx  \
mean  0.0      -0.0       0.0    0.0      -0.0          -0.0            -0.0   
std   1.0       1.0       1.0    1.0       1.0           1.0             1.0   

      cons.conf.idx  euribor3m  nr.employed  
mean            0.0        0.0          0.0  
std             1.0        1.0          1.0  


In [15]:
df.head()

,age,education,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,1.576735,1.0,0.100728,-0.822641,0.195458,-0.34958,0.648073,0.72254,0.886533,0.712464,...,1,0,0,0,1,0,0,0,1,0
1,1.675004,4.0,-0.478303,-0.822641,0.195458,-0.34958,0.648073,0.72254,0.886533,0.712464,...,1,0,0,0,1,0,0,0,1,0
2,-0.290377,4.0,-0.080219,-0.822641,0.195458,-0.34958,0.648073,0.72254,0.886533,0.712464,...,1,0,0,0,1,0,0,0,1,0
3,0.004430,2.0,-0.467963,-0.822641,0.195458,-0.34958,0.648073,0.72254,0.886533,0.712464,...,1,0,0,0,1,0,0,0,1,0
4,1.576735,4.0,0.338544,-0.822641,0.195458,-0.34958,0.648073,0.72254,0.886533,0.712464,...,1,0,0,0,1,0,0,0,1,0


In [17]:
# Guardar el dataset limpio para el modelo
df.to_csv('../data/interim/03-bank_marketing_cleaned.csv', index=False)